In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

# output directory
sub_out = "chaphama_260206_MSigDB_Hallmark_2020_stat_blastx_filtered/"
out_folder_path = f"{basedir}/output/250909_4474_samples/final_test/"
out_path = out_folder_path + sub_out
plot_path = out_path + "heatmap_chaphama/"

# input file
input_folder_path = out_path
# gene table path
gene_table_path = f"{basedir}/data/prompt3_selected_genes_long_with_interpretation.tsv"
# RNA seq
input_rna_path = f"{basedir}/data/251215_rna_seq/To_kurihara_downloaded260209_chaphama/output/"

# make directories
if not(os.path.exists(out_folder_path)):
    os.mkdir(out_folder_path)
if not(os.path.exists(out_path)):
    os.mkdir(out_path)
if not(os.path.exists(plot_path)):
    os.mkdir(plot_path)

print("saving files to:", plot_path)

saving files to: /Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/250909_4474_samples/final_test/chaphama_260206_MSigDB_Hallmark_2020_stat_blastx_filtered/heatmap_chaphama/


# gene selection

In [3]:
# map term and genes
ttg = defaultdict(list)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]
        print("\nprocessing...", project)
        print("size:", res.shape)
        print("#NaN:", res.isna().sum().sum())
        print("#uniq Term:", len(set(res["Term"].to_list())))
        # print("dtypes:\n", res.dtypes)
        
        # term list
        selected_terms = [
            "Epithelial Mesenchymal Transition",
            "G2-M Checkpoint",
            "Interferon Alpha Response",
            "Allograft Rejection",
            "Inflammatory Response",
            "IL-6/JAK/STAT3 Signaling",
            "Xenobiotic Metabolism",
            "Bile Acid Metabolism",
            "Fatty Acid Metabolism",
            "Adipogenesis"
        ]

        # map term and lead genes
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            print(term)
            genes = res.loc[idx, "Lead_genes"].split(";")
            if len(genes) != len(set(genes)):
                print("genes not unique")
            for gene in genes:
                if gene not in ttg[term]:
                    ttg[term].append(gene)


processing... PRJNA622813
size: (50, 10)
#NaN: 0
#uniq Term: 50
Interferon Alpha Response
Allograft Rejection
G2-M Checkpoint
Inflammatory Response
IL-6/JAK/STAT3 Signaling
Adipogenesis
Fatty Acid Metabolism
Bile Acid Metabolism
Epithelial Mesenchymal Transition
Xenobiotic Metabolism

processing... PRJNA612882
size: (50, 10)
#NaN: 0
#uniq Term: 50
Interferon Alpha Response
Epithelial Mesenchymal Transition
Xenobiotic Metabolism
Allograft Rejection
G2-M Checkpoint
Bile Acid Metabolism
Inflammatory Response
Fatty Acid Metabolism
IL-6/JAK/STAT3 Signaling
Adipogenesis

processing... PRJNA577590
size: (50, 10)
#NaN: 0
#uniq Term: 50
G2-M Checkpoint
Interferon Alpha Response
Adipogenesis
IL-6/JAK/STAT3 Signaling
Xenobiotic Metabolism
Bile Acid Metabolism
Fatty Acid Metabolism
Epithelial Mesenchymal Transition
Inflammatory Response
Allograft Rejection


In [4]:
# create table
df = pd.DataFrame(
    [(k, v) for k, vs in ttg.items() for v in vs],
    columns=["term", "gene"]
)
# add group
group = {
    "Epithelial Mesenchymal Transition": 1,
    "G2-M Checkpoint": 2,
    "Interferon Alpha Response": 3,
    "Allograft Rejection": 4,
    "Inflammatory Response": 5,
    "IL-6/JAK/STAT3 Signaling": 5,
    "Xenobiotic Metabolism": 6,
    "Bile Acid Metabolism": 6,
    "Fatty Acid Metabolism": 7,
    "Adipogenesis": 7
}
df["group"] = df["term"].apply(lambda x: group[x])
df.to_csv(plot_path + "term2gene.tsv" ,sep="\t")
# show
print(df.shape)
print(df.isna().sum().sum())
df.head(3)

(884, 3)
0


,term,gene,group
0,Interferon Alpha Response,DDX60,3
1,Interferon Alpha Response,USP18,3
2,Interferon Alpha Response,RSAD2,3


# Creates table for heatmap

In [5]:
## show gene df
genedf = pd.read_table(gene_table_path, skiprows=1)
genedf

,group,term,direction,gene,interpretation
0,1,Epithelial Mesenchymal Transition,UP,COL1A1,I型コラーゲン；線維化/間質増生の代表的マーカー。
1,1,Epithelial Mesenchymal Transition,UP,FN1,細胞外マトリクス（フィブロネクチン）；組織再構築/線維化で上昇しやすい。
2,1,Epithelial Mesenchymal Transition,UP,SNAI2,EMTを誘導する転写因子（Slug）；上皮性低下・線維化/浸潤性の指標。
3,1,Epithelial Mesenchymal Transition,UP,TGFB1,TGF-β1；線維化・EMT誘導の中心的サイトカイン。
4,1,Epithelial Mesenchymal Transition,UP,VIM,EMT/線維化で上がりやすい間葉系マーカー（ビメンチン）。
5,2,G2-M Checkpoint,UP,AURKA,紡錘体形成に関わるキナーゼ；分裂活性/増殖の指標。
6,2,G2-M Checkpoint,UP,CDC20,APC/C活性化因子；有糸分裂チェックポイント/増殖の指標。
7,2,G2-M Checkpoint,UP,CDK1,G2/M移行の必須キナーゼ；増殖・再生/腫瘍性増殖シグナルの指標。
8,2,G2-M Checkpoint,UP,PLK1,有糸分裂進行キナーゼ；細胞増殖・分裂活性の指標。
9,2,G2-M Checkpoint,UP,TOP2A,DNAトポイソメラーゼIIα；増殖細胞で高発現しやすい。


In [6]:
# check if all genes exist
# map and term and genes
test_dic = defaultdict(set)
all_genes = set()

# read gene table
genedf = pd.read_table(gene_table_path, skiprows=1)
genes_oi = []
for g in genedf["gene"].to_list():
    if g not in genes_oi:
        genes_oi.append(g)

t_order = []
for t in genedf["term"].to_list():
    if t not in t_order:
        t_order.append(t)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # only selected term
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            genes = res.loc[idx, "Lead_genes"].split(";")

            # get gene info
            for gene in genes:
                all_genes.add(gene)
                 # get term info per gene
                if gene not in genes_oi:
                    continue
                test_dic[gene].add(term)

# check if genes in table exist
print(f"\n#all genes: {len(all_genes)}")
for goi in genes_oi:
    if goi not in sorted(list(all_genes)):
        print(f"### {goi} not exist! ###")
# - Yes

# check if genes have uniq term
testdf = pd.DataFrame(
    [(k, v) for k, vs in test_dic.items() for v in sorted(list(vs))],
    columns=["gene", "term"]
)
# - No

# term anno df
term_bin = (pd.crosstab(testdf["gene"], testdf["term"]) > 0).astype(int)
term_bin = term_bin[t_order]
term_bin.columns = term_bin.columns.map(lambda x: x.replace("-", "_"))
# sort
term_bin = term_bin.sort_values(
            "gene",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
# save
term_bin.to_csv(plot_path + "term_anno.csv")

# show
term_bin.head(3)


#all genes: 761


term,Epithelial Mesenchymal Transition,G2_M Checkpoint,Interferon Alpha Response,Allograft Rejection,IL_6/JAK/STAT3 Signaling,Inflammatory Response,Bile Acid Metabolism,Xenobiotic Metabolism,Adipogenesis,Fatty Acid Metabolism
gene,,,,,,,,,,
COL1A1,1,0,0,0,0,0,0,0,0,0
FN1,1,0,0,0,0,0,0,0,0,0
SNAI2,1,0,0,0,0,0,0,0,0,0


In [7]:
# obtain pivot table
dfs = []
for dirpath, dirnames, filenames in os.walk(input_rna_path):
    for fname in filenames:
        if not(fname.endswith("_hs_gallus_and_Gene.txt")):
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)

        # extract genes
        df_gene = project.loc[project["gene_name_hs"].isin(genes_oi), ["gene_name_hs", "log2FoldChange"]].reset_index(drop=True).copy()
        df_gene["Project"] = proname
        print(f"size for {proname}: {df_gene.shape}")

        # append
        dfs.append(df_gene)

# concat
df_cat = pd.concat(dfs)
print("\nsize for all:", df_cat.shape)
print(df_cat.head(3))
# pivot
df_pivot = (df_cat.pivot_table(index="gene_name_hs", columns="Project", values="log2FoldChange").sort_index())
print("\nsize for df pivot:", df_pivot.shape)
print("#NaN for df pivot:", df_pivot.isna().sum().sum())
# order
df_pivor_sorted = df_pivot.sort_values(
            "gene_name_hs",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
df_pivor_sorted = df_pivor_sorted[["PRJNA622813", "PRJNA612882", "PRJNA577590"]]
print("size for sorted:", df_pivor_sorted.shape)
# save
df_pivor_sorted.to_csv(plot_path + "gene_project_log2FC_matrix.csv")
# show
df_pivor_sorted.head(3)


processing... PRJNA577590
size for PRJNA577590: (49, 3)

processing... PRJNA622813
size for PRJNA622813: (49, 3)

processing... PRJNA612882
size for PRJNA612882: (49, 3)

size for all: (147, 3)
  gene_name_hs  log2FoldChange      Project
0       AKR1D1        0.896767  PRJNA577590
1        USP18        1.097369  PRJNA577590
2        MYD88        0.272963  PRJNA577590

size for df pivot: (49, 3)
#NaN for df pivot: 0
size for sorted: (49, 3)


Project,PRJNA622813,PRJNA612882,PRJNA577590
gene_name_hs,,,
COL1A1,-0.050896,0.461615,-0.021546
FN1,0.912364,-0.066378,0.878729
SNAI2,0.587300,0.412287,-1.302434
